In [2]:
from google.colab import drive
drive.mount('/content/drive')
!apt-get -qq install -y tesseract-ocr
!pip -q install opencv-python-headless pytesseract openpyxl pillow numpy

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!python /content/drive/MyDrive/CDS_Audit/cds_audit_ocr_colab.py --evaluate-only

CDS Audit - Accuracy Report

--- Session ID ---
Rows evaluated: 40
Accuracy: 17.5%
Confusion Matrix (rows = predicted, cols = actual):
                    Actual Wrong   Actual Correct
  Flagged (Review)   TP=6          FP=0           <- Type I errors
  Not Flagged        FN=33         TN=1           <- Type II errors
Type I error rate  (correct rows wrongly flagged for review): 0.0%
Type II error rate (wrong rows that slipped through unflagged): 84.6%

--- Date/Time ---
Rows evaluated: 40
Accuracy: 57.5%
Confusion Matrix (rows = predicted, cols = actual):
                    Actual Wrong   Actual Correct
  Flagged (Review)   TP=2          FP=0           <- Type I errors
  Not Flagged        FN=17         TN=21          <- Type II errors
Type I error rate  (correct rows wrongly flagged for review): 0.0%
Type II error rate (wrong rows that slipped through unflagged): 89.5%

Accuracy Report sheet added to: /content/drive/MyDrive/CDS_Audit/CDS_Audit_Results.xlsx


In [3]:
"""
CDS Receipt Audit - OCR Extraction Pipeline (COLAB + GOOGLE DRIVE VERSION)
============================================================================
Extracts Receipt No. (handwritten), Session ID, and Date/Time from photographed
CDS (Container Deposit Scheme) receipts, and writes everything to an Excel file
with embedded verification thumbnails.

WHAT CHANGED IN THIS VERSION
-----------------------------
1. Runs in Google Colab, reading/writing straight from Google Drive.
2. Session ID and Date/Time each now get their own dedicated, high-resolution,
   character-whitelisted crop + multi-PSM OCR ensemble (previously Date/Time
   was only read from the noisy full-page OCR pass, which is why it came out
   garbled). Receipt No. and everything else is untouched, as requested.
3. Adds "Session ID Correct? (Y/N)" and "Date/Time Correct? (Y/N)" columns for
   you to fill in by eye against the embedded zoomed images. Once filled,
   re-run the last cell (or call evaluate_accuracy()) to get Accuracy, a
   Confusion Matrix, and Type I / Type II error rates.
   NOTE: true accuracy requires a human-verified ground truth - OCR cannot
   grade its own correctness. These columns are how you supply that ground
   truth, and it only takes a quick glance at each thumbnail.

HOW TO RUN IN COLAB
--------------------
Cell 1:
    from google.colab import drive
    drive.mount('/content/drive')
    !apt-get -qq install -y tesseract-ocr
    !pip -q install opencv-python-headless pytesseract openpyxl pillow numpy

Cell 2:
    Edit the three paths in the CONFIG section below (INPUT_FOLDER,
    OUTPUT_EXCEL, TEMP_CROPS_FOLDER), then run this whole file, e.g.:
    %run /content/drive/MyDrive/CDS_Audit/cds_audit_ocr_colab.py

Cell 3 (after you've filled in the Correct? Y/N columns in the Excel):
    %run /content/drive/MyDrive/CDS_Audit/cds_audit_ocr_colab.py --evaluate-only
"""

import os
import re
import sys
import json
from collections import Counter

import cv2
import numpy as np
import pytesseract
from PIL import Image as PILImage
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

# =====================================================================
# CONFIG - edit these three paths for your Google Drive
# =====================================================================

# Folder containing your receipt photos (inside Google Drive). Can have
# subfolders (e.g. "day 1 batches", "day 2 batches") - every .jpg/.jpeg/.png
# found anywhere under this folder will be processed, and its immediate
# parent folder name is used as the "Batch" label in the output.
INPUT_FOLDER = "/content/drive/MyDrive/CDS Audit Batches Required"

# Where to save the final Excel file
OUTPUT_EXCEL = "/content/drive/MyDrive/CDS_Audit/CDS_Audit_Results.xlsx"

# Scratch folder for intermediate crop images (created automatically)
TEMP_CROPS_FOLDER = "/content/drive/MyDrive/CDS_Audit/temp_crops"

# On Colab/Linux, tesseract is on PATH after `apt-get install tesseract-ocr`.
# Only change this if `!which tesseract` in Colab returns something different.
pytesseract.pytesseract.tesseract_cmd = "tesseract"


# =====================================================================
# STAGE 1: LOCATE + DESKEW THE RECEIPT (separate it from the background)
# =====================================================================

def _foreground_mask(gray):
    blur = cv2.GaussianBlur(gray, (15, 15), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m255 = gray[th == 255].mean() if (th == 255).any() else 0
    m0 = gray[th == 0].mean() if (th == 0).any() else 0
    if m0 > m255:
        th = 255 - th
    return th


def _paper_bbox_by_projection(mask):
    """Robust to localized glare/reflections: the true receipt spans almost the
    full photo height, so find columns that are foreground for most of that
    height (glare blobs are usually localized vertically and fail this test)."""
    h, w = mask.shape
    col_frac = (mask > 0).sum(axis=0) / float(h)
    paper_cols = col_frac > 0.55
    runs, start = [], None
    for i, v in enumerate(paper_cols):
        if v and start is None:
            start = i
        elif not v and start is not None:
            runs.append((start, i))
            start = None
    if start is not None:
        runs.append((start, w))
    if not runs:
        return 0, 0, w, h
    x0, x1 = max(runs, key=lambda r: r[1] - r[0])

    sub = mask[:, x0:x1]
    row_frac = (sub > 0).sum(axis=1) / float(x1 - x0)
    paper_rows = row_frac > 0.5
    rruns, start = [], None
    for i, v in enumerate(paper_rows):
        if v and start is None:
            start = i
        elif not v and start is not None:
            rruns.append((start, i))
            start = None
    if start is not None:
        rruns.append((start, h))
    y0, y1 = max(rruns, key=lambda r: r[1] - r[0]) if rruns else (0, h)
    return x0, y0, x1, y1


def _best_receipt_contour(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    return max(contours, key=cv2.contourArea)


def locate_and_deskew(img):
    """Find the receipt strip in a cluttered background, deskew it upright,
    and return a tightly cropped BGR image of just the receipt."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = _foreground_mask(gray)

    pad = 40
    x0, y0, x1, y1 = _paper_bbox_by_projection(mask)
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(img.shape[1], x1 + pad); y1 = min(img.shape[0], y1 + pad)
    coarse = img[y0:y1, x0:x1]

    gray_c = cv2.cvtColor(coarse, cv2.COLOR_BGR2GRAY)
    mask_c = _foreground_mask(gray_c)
    kernel = np.ones((15, 15), np.uint8)
    mask_c = cv2.morphologyEx(mask_c, cv2.MORPH_CLOSE, kernel)
    c = _best_receipt_contour(mask_c)
    if c is None:
        return coarse, 0.0
    rect = cv2.minAreaRect(c)
    (cx, cy), (w, h), angle = rect
    if w > h:
        angle = angle + 90
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    rotated = cv2.warpAffine(coarse, M, (coarse.shape[1], coarse.shape[0]),
                              flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    gray2 = cv2.cvtColor(rotated, cv2.COLOR_BGR2GRAY)
    mask2 = _foreground_mask(gray2)
    mask2 = cv2.morphologyEx(mask2, cv2.MORPH_CLOSE, kernel)
    c2 = _best_receipt_contour(mask2)
    if c2 is None:
        c2 = c
    x, y, w2, h2 = cv2.boundingRect(c2)
    tpad = 12
    x = max(0, x - tpad); y = max(0, y - tpad)
    w2 = min(rotated.shape[1] - x, w2 + 2 * tpad)
    h2 = min(rotated.shape[0] - y, h2 + 2 * tpad)
    crop = rotated[y:y + h2, x:x + w2]
    return crop, angle


# =====================================================================
# STAGE 2: ENHANCE FOR OCR (illumination, denoise, sharpen, binarize)
# =====================================================================

def enhance_for_ocr(bgr_crop, upscale=2.0):
    gray = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2GRAY)
    if upscale != 1.0:
        gray = cv2.resize(gray, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
    flat = clahe.apply(gray)

    denoised = cv2.bilateralFilter(flat, d=7, sigmaColor=35, sigmaSpace=35)

    gauss = cv2.GaussianBlur(denoised, (0, 0), 2.0)
    sharp = cv2.addWeighted(denoised, 1.6, gauss, -0.6, 0)

    bin_adapt = cv2.adaptiveThreshold(sharp, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 31, 10)
    kernel = np.ones((2, 2), np.uint8)
    bin_adapt = cv2.morphologyEx(bin_adapt, cv2.MORPH_CLOSE, kernel)

    _, bin_otsu = cv2.threshold(sharp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return {"adaptive": bin_adapt, "otsu": bin_otsu, "gray_sharp": sharp}


def ocr_text(bin_img, psm=4):
    cfg = f"--oem 1 --psm {psm}"
    return pytesseract.image_to_string(bin_img, config=cfg)


# =====================================================================
# STAGE 3: LOCATE THE SESSION-ID LINE(S) *AND* THE DATE/TIME LINE
# =====================================================================
# Both regions are located from a single Tesseract word/line pass (faster
# than doing two separate low-res OCR passes), then each is re-cropped and
# re-OCR'd separately at much higher resolution with a character whitelist
# tailored to what can legally appear in that field. Restricting the
# whitelist is what fixes most of the garbled characters: Tesseract can only
# ever output digits/hyphens for the Session ID, and only digits/"/"/":" for
# the Date/Time, so it can no longer hallucinate letters or punctuation.

def _cluster_words_into_lines(crop_bgr, upscale=1.8):
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(gray, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)
    _, otsu = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    data = pytesseract.image_to_data(otsu, config="--oem 1 --psm 4",
                                      output_type=pytesseract.Output.DICT)
    n = len(data["text"])
    words = []
    for i in range(n):
        t = data["text"][i]
        if t.strip():
            words.append({"text": t, "top": data["top"][i], "height": data["height"][i],
                           "left": data["left"][i], "width": data["width"][i]})
    if not words:
        return [], g
    words.sort(key=lambda w: w["top"])
    med_h = sorted(w["height"] for w in words)[len(words) // 2]

    lines = []
    current = [words[0]]
    for w in words[1:]:
        if w["top"] - current[-1]["top"] < med_h * 0.6:
            current.append(w)
        else:
            lines.append(current)
            current = [w]
    lines.append(current)
    return lines, g


def _line_bbox(line):
    top = min(w["top"] for w in line)
    bottom = max(w["top"] + w["height"] for w in line)
    left = min(w["left"] for w in line)
    right = max(w["left"] + w["width"] for w in line)
    return left, top, right, bottom


def _box_from_lines(lines_subset, g_shape, upscale, pad_px=22):
    left, top, right, bottom = _line_bbox([w for line in lines_subset for w in line])
    pad = int(pad_px * upscale)
    top = max(0, top - pad); left = max(0, left - pad)
    bottom = min(g_shape[0], bottom + pad); right = min(g_shape[1], right + pad)
    return (int(left / upscale), int(top / upscale),
            int(right / upscale), int(bottom / upscale))


def locate_key_regions(crop_bgr, upscale=1.8):
    """Returns (session_box, datetime_box), each as (left, top, right, bottom)
    in original (non-upscaled) crop coordinates, or None if not found."""
    lines, g = _cluster_words_into_lines(crop_bgr, upscale=upscale)
    if not lines:
        return None, None

    # ---- Session ID: line with a long hyphen-heavy token (the UUID itself)
    session_idx = None
    for idx, line in enumerate(lines):
        for w in line:
            if len(w["text"]) >= 15 and w["text"].count("-") >= 2:
                session_idx = idx
                break
        if session_idx is not None:
            break
    if session_idx is None:
        for idx, line in enumerate(lines):
            if any("sess" in w["text"].lower() for w in line):
                session_idx = idx + 1  # UUID starts on the *next* visual line
                break

    session_box = None
    session_line_span = 0
    if session_idx is not None and 0 <= session_idx < len(lines):
        subset = [lines[session_idx]]
        # include one more wrap-around line if it also looks hex/hyphen-ish
        if session_idx + 1 < len(lines):
            nxt_text = "".join(w["text"] for w in lines[session_idx + 1])
            nxt_clean = re.sub(r"[^0-9A-Za-z\-]", "", nxt_text)
            if len(nxt_clean) >= 6:
                subset.append(lines[session_idx + 1])
        session_line_span = len(subset)
        session_box = _box_from_lines(subset, g.shape, upscale)
    else:
        # tertiary: anchor on "Machine #CNT-1"
        for idx, line in enumerate(lines):
            if any("machin" in w["text"].lower() for w in line):
                _, _, _, mbottom = _line_bbox(line)
                top = mbottom + int(15 * upscale)
                bottom = top + int(330 * upscale)
                session_box = (0, int(top / upscale), g.shape[1], int(bottom / upscale))
                session_idx = idx
                session_line_span = 2
                break
    if session_box is None:
        h, w = g.shape
        session_box = (0, int(h * 0.29 / upscale), int(w / upscale), int(h * 0.40 / upscale))

    # ---- Date/Time: prefer a line (after the session block) with a lot of
    # digits plus "/" or ":" in it; fall back to the line positionally right
    # after the session block; last resort, a proportional band.
    datetime_box = None
    search_start = (session_idx + session_line_span) if session_idx is not None else 0
    for idx in range(search_start, len(lines)):
        text = "".join(w["text"] for w in lines[idx])
        digits = sum(c.isdigit() for c in text)
        if digits >= 6 and ("/" in text or ":" in text or "." in text):
            datetime_box = _box_from_lines([lines[idx]], g.shape, upscale)
            break
    if datetime_box is None and session_idx is not None and search_start < len(lines):
        # positional fallback: the very next visual line after the session block
        datetime_box = _box_from_lines([lines[search_start]], g.shape, upscale)
    if datetime_box is None:
        h, w = g.shape
        datetime_box = (0, int(h * 0.40 / upscale), int(w / upscale), int(h * 0.46 / upscale))

    return session_box, datetime_box


def ocr_line_candidates(line_bgr_or_gray, whitelist, scale=5.0, psms=(7, 6, 4)):
    """Run several preprocessing+psm combos over a single tight line crop,
    restricted to `whitelist` characters, and return all raw text candidates
    for ensemble voting."""
    if len(line_bgr_or_gray.shape) == 3:
        gray = cv2.cvtColor(line_bgr_or_gray, cv2.COLOR_BGR2GRAY)
    else:
        gray = line_bgr_or_gray
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    flat = clahe.apply(gray)
    den = cv2.bilateralFilter(flat, 7, 35, 35)
    gauss = cv2.GaussianBlur(den, (0, 0), 2.0)
    sharp = cv2.addWeighted(den, 1.6, gauss, -0.6, 0)

    _, otsu = cv2.threshold(sharp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adapt = cv2.adaptiveThreshold(sharp, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 31, 10)
    # extra variant: mild dilation helps thin/broken thermal-print strokes
    dilated = cv2.dilate(otsu, np.ones((2, 2), np.uint8), iterations=1)

    candidates = []
    for variant_name, variant in [("otsu", otsu), ("adapt", adapt), ("dilated", dilated)]:
        for psm in psms:
            cfg = f"--oem 1 --psm {psm} -c tessedit_char_whitelist={whitelist}"
            txt = pytesseract.image_to_string(variant, config=cfg).strip()
            candidates.append((f"{variant_name}_psm{psm}", txt))
    return candidates


# =====================================================================
# STAGE 4a: SESSION-ID FIELD EXTRACTION + UUID-AWARE CHARACTER CORRECTION
# =====================================================================

SESSION_LOOSE_RE = re.compile(
    r"[Ss]ession[^\n]*\n?\s*([0-9A-Za-z]{6,9}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{10,14})"
)
UUID_SHAPE_RE = re.compile(
    r"([0-9A-Za-z]{6,9}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{10,14})"
)
LEADING_UUID_RE = re.compile(
    r"^([0-9A-Za-z]{6,10}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{3,5}-[0-9A-Za-z]{10,14})"
)
CONFUSABLE = set("cefl1")  # the genuinely ambiguous characters in this font


def _extract_uuid_lines(raw_text):
    """From a noisy multi-line OCR result, keep only the lines that plausibly
    belong to the UUID itself: a line is kept if it looks hyphen-heavy/hex-like
    enough on its own, OR if it immediately follows a kept line (the UUID's
    own un-hyphenated wrap-around continuation). Lines containing '/' or ':'
    (the date) are always dropped."""
    kept, keep_next = [], False
    for line in raw_text.splitlines():
        if "/" in line or ":" in line:
            keep_next = False
            continue
        s = re.sub(r"[^0-9A-Za-z\-]", "", line)
        if not s:
            continue
        looks_like_uuid_piece = len(s) >= 15 and s.count("-") >= 2
        is_continuation = keep_next and len(s) >= 6
        if looks_like_uuid_piece or is_continuation:
            kept.append(s)
            keep_next = True
        else:
            keep_next = False
    return "".join(kept)


def best_session_id_guess(candidates):
    """Pick the candidate string that best matches the expected UUID shape;
    among equal-shape candidates, prefer the one repeated most often."""
    cleaned = []
    for _, txt in candidates:
        s = _extract_uuid_lines(txt)
        s = re.sub(r"-{2,}", "-", s)
        m = LEADING_UUID_RE.match(s)
        if m:
            s = m.group(1)
        if s:
            cleaned.append(s)
    if not cleaned:
        return None, []

    def shape_score(s):
        parts = s.split("-")
        target = [8, 4, 4, 4, 12]
        if len(parts) != 5:
            return -abs(len(s) - 36)
        return -sum(abs(len(p) - t) for p, t in zip(parts, target))

    cleaned.sort(key=shape_score, reverse=True)
    best_shape = shape_score(cleaned[0])
    finalists = [s for s in cleaned if shape_score(s) == best_shape]
    winner = Counter(finalists).most_common(1)[0][0]
    return winner, cleaned


def fix_uuid_charset(raw):
    """Force the result onto the legal UUID alphabet [0-9a-f -], auto-correcting
    characters that are unambiguous in this font, and flagging the rest."""
    if raw is None:
        return None, []
    s = raw.strip().lower()
    s = re.sub(r"[^0-9a-z\-]", "", s)
    flags, fixed_chars = [], []
    for i, ch in enumerate(s):
        if ch == "-":
            fixed_chars.append(ch)
            continue
        if ch not in "0123456789abcdef":
            mapping = {"o": "0", "l": "1", "i": "1", "s": "5", "g": "9",
                       "z": "2", "q": "9", "u": "0", "b": "b"}
            newch = mapping.get(ch, ch)
            flags.append(f"pos{i}:'{ch}'->'{newch}'(forced)")
            fixed_chars.append(newch)
        else:
            fixed_chars.append(ch)
            if ch in CONFUSABLE:
                flags.append(f"pos{i}:'{ch}'(verify - c/e/f/1/l risk)")
    return "".join(fixed_chars), flags


def session_id_plausible(s):
    """Reject results that are clearly OCR noise (one character dominating
    the whole string, or wildly wrong length) rather than a genuine
    low-confidence-but-real reading."""
    if not s:
        return False
    core = s.replace("-", "")
    if len(core) < 24 or len(core) > 34:
        return False
    most_common_count = Counter(core).most_common(1)[0][1]
    if most_common_count / len(core) > 0.35:
        return False
    return True


# =====================================================================
# STAGE 4b: DATE/TIME FIELD EXTRACTION (dedicated zoomed crop + digit-only OCR)
# =====================================================================

# Tolerant parser: allows separators to be missing/garbled or glued together
# (common with thermal-receipt fonts at small size), since it anchors on the
# fixed digit-group lengths of DD/M/YYYY HH:MM:SS rather than the literal
# punctuation.
DATE_TIME_FLEX_RE = re.compile(
    r"(\d{1,2})\D{0,2}(\d{1,2})\D{0,2}(\d{4})\D{0,3}(\d{1,2})\D{0,2}(\d{2})\D{0,2}(\d{2})"
)
# Used only as a fallback against full-page OCR text (kept from original script)
DATE_RE = re.compile(r"(\d{1,2}[/.]\d{1,2}[/.]\d{4})\s+(\d{1,2}:\d{2}:\d{2})")


def _parse_datetime_flex(text):
    m = DATE_TIME_FLEX_RE.search(text)
    if not m:
        return None, None
    d, mo, y, h, mi, s = m.groups()
    if not (1 <= int(mo) <= 12) or not (1 <= int(d) <= 31):
        return None, None
    if not (0 <= int(h) <= 23) or not (0 <= int(mi) <= 59) or not (0 <= int(s) <= 59):
        return None, None
    date_str = f"{int(d)}/{int(mo)}/{y}"
    time_str = f"{int(h):02d}:{mi}:{s}"
    return date_str, time_str


def best_datetime_guess(candidates):
    """Ensemble-vote across OCR candidates; prefer the (date,time) pair that
    appears most often among candidates that parse cleanly."""
    parsed = []
    for _, txt in candidates:
        d, t = _parse_datetime_flex(txt.replace(" ", ""))
        if d and t:
            parsed.append((d, t))
    if not parsed:
        return None, None
    winner, _count = Counter(parsed).most_common(1)[0]
    return winner


def datetime_plausible(date, time):
    if not date or not time:
        return False
    return bool(re.match(r"^\d{1,2}/\d{1,2}/\d{4}$", date)) and bool(re.match(r"^\d{2}:\d{2}:\d{2}$", time))


def extract_fields_from_fulltext(full_text):
    """Fallback only: pull date/time and a rough session id out of the noisy
    full-page OCR text, used if the dedicated zoomed-crop pass fails outright."""
    result = {"date": None, "time": None, "session_id_raw": None}
    dm = DATE_RE.search(full_text)
    if dm:
        result["date"], result["time"] = dm.group(1), dm.group(2)
    else:
        d, t = _parse_datetime_flex(full_text.replace(" ", ""))
        if d:
            result["date"], result["time"] = d, t
    sm = SESSION_LOOSE_RE.search(full_text) or UUID_SHAPE_RE.search(full_text)
    if sm:
        result["session_id_raw"] = sm.group(1)
    return result


# =====================================================================
# STAGE 0: IMAGE QUALITY GRADING
# =====================================================================

def grade_image_quality(bgr_crop):
    gray = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2GRAY)
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    contrast_score = float(gray.std())
    brightness = float(gray.mean())
    issues = []
    if blur_score < 150:
        issues.append("blurry")
    if contrast_score < 40:
        issues.append("low contrast")
    if brightness < 80:
        issues.append("dark")
    if brightness > 230:
        issues.append("overexposed/glare")
    if not issues:
        grade = "Good"
    elif len(issues) == 1:
        grade = "Fair (" + issues[0] + ")"
    else:
        grade = "Poor (" + ", ".join(issues) + ")"
    return {"grade": grade, "blur_score": round(blur_score, 1),
            "contrast_score": round(contrast_score, 1), "brightness": round(brightness, 1)}


# =====================================================================
# STAGE 5: HANDWRITTEN RECEIPT NUMBER (TOP OF RECEIPT) - unchanged
# =====================================================================

def extract_receipt_no_region(crop_bgr, frac=0.075):
    h = crop_bgr.shape[0]
    return crop_bgr[0:int(h * frac), :]


def _find_ink_bbox(region_bgr):
    gray = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2GRAY)
    _, dark = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)
    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    sat_mask = cv2.threshold(hsv[:, :, 1], 60, 255, cv2.THRESH_BINARY)[1]
    ink = cv2.bitwise_or(dark, sat_mask)
    kernel = np.ones((5, 5), np.uint8)
    ink = cv2.morphologyEx(ink, cv2.MORPH_CLOSE, kernel)
    ink = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(ink, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = gray.shape
    boxes = []
    for c in contours:
        x, y, bw, bh = cv2.boundingRect(c)
        if bw * bh < 40:
            continue
        if x < 3 or x + bw > w - 3:  # leftover background sliver, not handwriting
            continue
        boxes.append((x, y, bw, bh))
    if not boxes:
        return None
    x0 = min(b[0] for b in boxes); y0 = min(b[1] for b in boxes)
    x1 = max(b[0] + b[2] for b in boxes); y1 = max(b[1] + b[3] for b in boxes)
    return x0, y0, x1, y1


def ocr_receipt_no(region_bgr):
    bbox = _find_ink_bbox(region_bgr)
    if bbox is None:
        return None, "(no handwriting detected in top region)"
    x0, y0, x1, y1 = bbox
    pad = 15
    h, w = region_bgr.shape[:2]
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(w, x1 + pad); y1 = min(h, y1 + pad)
    ink_crop = region_bgr[y0:y1, x0:x1]

    gray = cv2.cvtColor(ink_crop, cv2.COLOR_BGR2GRAY)
    scale = min(max(1.0, 200.0 / max(gray.shape[0], 1)), 10.0)
    gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    flat = clahe.apply(gray)
    _, otsu = cv2.threshold(flat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    free_text = pytesseract.image_to_string(otsu, config="--oem 1 --psm 7").strip()
    digits_only = pytesseract.image_to_string(
        otsu, config="--oem 1 --psm 7 -c tessedit_char_whitelist=0123456789()").strip()
    nums = re.findall(r"\d+", digits_only) or re.findall(r"\d+", free_text)
    guess = max(nums, key=len) if nums else None
    raw = free_text if free_text else "(unrecognized handwriting)"
    return guess, raw


# =====================================================================
# MAIN: PROCESS ALL IMAGES
# =====================================================================

def find_all_images(root):
    out = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                batch_label = os.path.basename(dirpath) or "root"
                out.append((dirpath, batch_label, fn))
    out.sort(key=lambda t: (t[1], t[2]))
    return out


def process_one_image(dirpath, batch_label, fname, out_crops_dir):
    path = os.path.join(dirpath, fname)
    img = cv2.imread(path)
    if img is None:
        return {"batch": batch_label, "file": fname, "error": "could not read image"}

    try:
        crop, angle = locate_and_deskew(img)
    except Exception as e:
        crop, angle = img, 0.0
        print("  deskew failed:", e)

    quality = grade_image_quality(crop)
    base = os.path.splitext(fname)[0]
    th_h = 260
    th_w = int(crop.shape[1] * th_h / crop.shape[0])
    thumb_path = os.path.join(out_crops_dir, f"{batch_label}_{base}_full.png")
    cv2.imwrite(thumb_path, cv2.resize(crop, (th_w, th_h), interpolation=cv2.INTER_AREA))

    # ---- locate both the Session ID block and the Date/Time line together ----
    session_box, datetime_box = None, None
    try:
        session_box, datetime_box = locate_key_regions(crop, upscale=1.8)
    except Exception as e:
        print("  region location failed:", e)

    # ---- session id: zoomed crop + whitelist ensemble ----
    session_raw, session_corrected, session_flags, session_crop_path = None, None, [], None
    if session_box:
        try:
            l, t, r, b = session_box
            line_img = crop[t:b, l:r]
            session_crop_path = os.path.join(out_crops_dir, f"{batch_label}_{base}_session.png")
            cv2.imwrite(session_crop_path, line_img)
            cands = ocr_line_candidates(line_img, whitelist="0123456789abcdefABCDEF-", scale=5.0)
            session_raw, _ = best_session_id_guess(cands)
            session_corrected, session_flags = fix_uuid_charset(session_raw)
        except Exception as e:
            print("  session extraction failed:", e)

    # ---- date/time: zoomed crop + digit-only whitelist ensemble ----
    date_val, time_val, datetime_crop_path = None, None, None
    if datetime_box:
        try:
            l, t, r, b = datetime_box
            dt_img = crop[t:b, l:r]
            datetime_crop_path = os.path.join(out_crops_dir, f"{batch_label}_{base}_datetime.png")
            cv2.imwrite(datetime_crop_path, dt_img)
            dt_cands = ocr_line_candidates(dt_img, whitelist="0123456789/.: ", scale=5.0)
            date_val, time_val = best_datetime_guess(dt_cands)
        except Exception as e:
            print("  datetime extraction failed:", e)

    # ---- fallback: noisy full-page OCR, only used if the dedicated pass above failed ----
    if not session_id_plausible(session_corrected) or not datetime_plausible(date_val, time_val):
        try:
            variants = enhance_for_ocr(crop, upscale=1.5)
            full_text = ocr_text(variants["otsu"], psm=4)
            fb = extract_fields_from_fulltext(full_text)
            if not datetime_plausible(date_val, time_val) and fb["date"]:
                date_val, time_val = fb["date"], fb["time"]
            if not session_id_plausible(session_corrected) and fb["session_id_raw"]:
                alt_corrected, alt_flags = fix_uuid_charset(fb["session_id_raw"])
                if session_id_plausible(alt_corrected):
                    session_corrected, session_flags = alt_corrected, alt_flags
                    session_raw = fb["session_id_raw"]
        except Exception as e:
            print("  full-page fallback OCR failed:", e)

    session_needs_review = not session_id_plausible(session_corrected)
    if session_needs_review:
        session_flags = session_flags + ["LOW CONFIDENCE - verify against embedded image"]
    datetime_needs_review = not datetime_plausible(date_val, time_val)

    # ---- receipt number (handwritten) - UNCHANGED ----
    receipt_no_guess, receipt_no_text, receipt_crop_path = None, None, None
    try:
        rno_region = extract_receipt_no_region(crop, frac=0.075)
        receipt_crop_path = os.path.join(out_crops_dir, f"{batch_label}_{base}_receiptno.png")
        cv2.imwrite(receipt_crop_path, rno_region)
        receipt_no_guess, receipt_no_text = ocr_receipt_no(rno_region)
    except Exception as e:
        print("  receipt no extraction failed:", e)

    return {
        "batch": batch_label, "file": fname,
        "receipt_no_guess": receipt_no_guess, "receipt_no_raw_text": receipt_no_text,
        "receipt_crop_path": receipt_crop_path,
        "session_id_raw": session_raw, "session_id_corrected": session_corrected,
        "session_id_flags": "; ".join(session_flags) if session_flags else "",
        "session_crop_path": session_crop_path, "thumb_path": thumb_path,
        "needs_manual_review": session_needs_review or datetime_needs_review,
        "session_needs_review": session_needs_review,
        "datetime_needs_review": datetime_needs_review,
        "date": date_val, "time": time_val,
        "datetime_crop_path": datetime_crop_path,
        "rotation_deg": round(angle, 2),
        "quality_grade": quality["grade"], "blur_score": quality["blur_score"],
        "contrast_score": quality["contrast_score"],
    }


def run_batch():
    os.makedirs(TEMP_CROPS_FOLDER, exist_ok=True)
    images = find_all_images(INPUT_FOLDER)
    if not images:
        raise SystemExit(f"No images found under {INPUT_FOLDER}")
    print(f"Found {len(images)} images. Starting...")

    records = []
    for i, (dirpath, batch_label, fname) in enumerate(images, start=1):
        print(f"[{i}/{len(images)}] {batch_label} / {fname}")
        rec = process_one_image(dirpath, batch_label, fname, TEMP_CROPS_FOLDER)
        records.append(rec)
        # incremental save so a crash partway through doesn't lose everything
        with open(os.path.join(TEMP_CROPS_FOLDER, "_records.json"), "w") as f:
            json.dump(records, f, indent=2)

    print(f"Done processing {len(records)} images.")
    return records


# =====================================================================
# EXCEL EXPORT
# =====================================================================

def build_excel(records, out_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "CDS Audit Results"

    HEADER_FILL = PatternFill("solid", start_color="1F4E78", end_color="1F4E78")
    HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    BASE_FONT = Font(name="Arial", size=10)
    REVIEW_FILL = PatternFill("solid", start_color="FFF2CC", end_color="FFF2CC")
    BAD_FILL = PatternFill("solid", start_color="FCE4E4", end_color="FCE4E4")
    GOOD_FILL = PatternFill("solid", start_color="E2F0D9", end_color="E2F0D9")
    THIN = Side(style="thin", color="BFBFBF")
    BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
    WRAP_TOP = Alignment(vertical="top", wrap_text=True)

    cols = [
        ("S.No", 6), ("Batch", 14), ("File Name", 22), ("Receipt Photo", 18),
        ("Receipt No. (OCR guess)", 16), ("Receipt No. Image", 26),
        ("Session ID (corrected)", 30), ("Session ID Image (zoomed)", 34),
        ("Char Flags / Notes", 38),
        ("Date", 12), ("Time", 11), ("Date/Time Image (zoomed)", 26),
        ("Image Quality", 22), ("Rotation Corrected (deg)", 10),
        ("Manual Review?", 14),
        ("Session ID Correct? (Y/N)", 14), ("Date/Time Correct? (Y/N)", 14),
    ]
    for idx, (name, width) in enumerate(cols, start=1):
        c = ws.cell(row=1, column=idx, value=name)
        c.font = HEADER_FONT
        c.fill = HEADER_FILL
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border = BORDER
        ws.column_dimensions[get_column_letter(idx)].width = width
    ws.row_dimensions[1].height = 30

    tmp_thumb_dir = os.path.join(TEMP_CROPS_FOLDER, "xlsx_thumbs")
    os.makedirs(tmp_thumb_dir, exist_ok=True)

    def add_image(path, cell_ref, max_h_px=125):
        if not path or not os.path.exists(path):
            return
        with PILImage.open(path) as im:
            im = im.convert("RGB")
            w, h = im.size
            scale = max_h_px / h
            new_size = (max(1, int(w * scale)), max_h_px)
            im_small = im.resize(new_size, PILImage.LANCZOS)
            tmp_path = os.path.join(tmp_thumb_dir, os.path.basename(path))
            im_small.save(tmp_path, "PNG", optimize=True)
        img = XLImage(tmp_path)
        img.height, img.width = new_size[1], new_size[0]
        ws.add_image(img, cell_ref)

    ROW_H = 100
    for i, r in enumerate(records, start=1):
        row = i + 1
        ws.row_dimensions[row].height = ROW_H
        vals = [
            i, r.get("batch", ""), r.get("file", ""), "",
            r.get("receipt_no_guess") or "", "",
            r.get("session_id_corrected") or "", "",
            r.get("session_id_flags") or "",
            r.get("date") or "", r.get("time") or "", "",
            r.get("quality_grade") or "",
            r.get("rotation_deg") if r.get("rotation_deg") is not None else "",
            "YES" if r.get("needs_manual_review") else "",
            "", "",
        ]
        for col_idx, v in enumerate(vals, start=1):
            cell = ws.cell(row=row, column=col_idx, value=v)
            cell.font = BASE_FONT
            cell.border = BORDER
            cell.alignment = WRAP_TOP

        add_image(r.get("thumb_path"), f"D{row}", max_h_px=128)
        add_image(r.get("receipt_crop_path"), f"F{row}", max_h_px=85)
        add_image(r.get("session_crop_path"), f"H{row}", max_h_px=85)
        add_image(r.get("datetime_crop_path"), f"L{row}", max_h_px=85)

        if r.get("needs_manual_review"):
            fill = REVIEW_FILL
        elif "Poor" in (r.get("quality_grade") or ""):
            fill = BAD_FILL
        else:
            fill = GOOD_FILL
        for col_idx in range(1, len(cols) + 1):
            if col_idx not in (4, 6, 8, 12):
                ws.cell(row=row, column=col_idx).fill = fill

        if not r.get("receipt_no_guess"):
            rc = ws.cell(row=row, column=5)
            rc.value = "verify manually"
            rc.font = Font(name="Arial", size=9, italic=True, color="999999")

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(cols))}{len(records) + 1}"

    notes = wb.create_sheet("Read Me - Methodology")
    title_font = Font(name="Arial", size=12, bold=True)
    section_font = Font(name="Arial", size=11, bold=True)
    notes_font = Font(name="Arial", size=10)
    lines = [
        ("CDS Audit Batches - OCR Extraction Methodology", title_font),
        ("", notes_font),
        ("Processing pipeline (OpenCV + Tesseract OCR):", section_font),
        ("1. Receipt location: the receipt strip is separated from the cluttered background "
         "using brightness-based segmentation plus a height-projection check (robust to glare/"
         "reflections touching the paper edge).", notes_font),
        ("2. Deskew: the paper's rotation angle is measured (minAreaRect) and corrected so all "
         "text is upright.", notes_font),
        ("3. Region location: a single word/line pass over the whole receipt finds where the "
         "Session ID block and the Date/Time line each sit (anchored on the UUID's own hyphen "
         "pattern and on digit+separator density respectively), with positional and proportional "
         "fallbacks if that pass is unreliable on a given photo.", notes_font),
        ("4. Session ID: the located region is re-cropped and re-OCR'd at 5x resolution with "
         "multiple binarization/PSM combinations, restricted to the legal UUID alphabet "
         "(0-9, a-f, '-'), voted for the best match to the UUID shape (8-4-4-4-12 hex characters).",
         notes_font),
        ("5. Date/Time: the located region is likewise re-cropped and re-OCR'd at 5x resolution, "
         "restricted to digits and '/','.',':' only (this whitelist is what fixes most of the "
         "garbled characters vs. reading it off the full page). A tolerant parser then reconstructs "
         "DD/M/YYYY HH:MM:SS even if a separator was dropped or misread, anchoring on the fixed "
         "digit-group lengths.", notes_font),
        ("6. Receipt No.: the handwritten mark at the top of each receipt is located by ink-color/"
         "darkness detection (not full-page OCR, which hallucinated on the mostly-blank area), "
         "then OCR'd. Handwriting recognition is inherently unreliable - treat this column as a "
         "starting guess only.", notes_font),
        ("7. Image quality grading: blur (Laplacian variance), contrast, and brightness are "
         "measured per photo and graded Good / Fair / Poor.", notes_font),
        ("", notes_font),
        ("Important - the c/e/f/1/l character ambiguity (Session ID only):", section_font),
        ("This receipt's font renders 'c' and 'e' as nearly identical open curves, and 'f', '1', "
         "and 'l' as nearly identical narrow vertical strokes. The Session ID is a UUID (characters "
         "0-9 and a-f only), which let us auto-correct some errors with certainty (e.g. any 'l' is "
         "definitely a mis-read '1', since 'l' cannot legally appear in a UUID). However 'c' vs 'e' "
         "and 'f' vs '1' are BOTH legal hex characters, so no automatic rule can resolve that "
         "ambiguity - it requires a human eye on the zoomed image.", notes_font),
        ("", notes_font),
        ("How to use this workbook:", section_font),
        ("- Every row includes a thumbnail of the full receipt, and zoomed crops of the "
         "handwritten receipt number, the Session ID, and the Date/Time line, so you can visually "
         "verify each OCR guess without reopening the original photo.", notes_font),
        ("- Rows shaded yellow ('Manual Review?' = YES) have an OCR result on Session ID and/or "
         "Date/Time that failed a basic plausibility check - read the embedded image directly for "
         "these.", notes_font),
        ("- Rows shaded red have 'Poor' image quality (blurry/low-contrast/dark/overexposed) - OCR "
         "results on every field in these rows carry higher risk and are worth a second look.", notes_font),
        ("- The 'Receipt No.' column is a best-effort handwriting read and should be treated as a "
         "draft, not a final value, for every row.", notes_font),
        ("", notes_font),
        ("Getting Accuracy / Confusion Matrix / Type I & II errors:", section_font),
        ("OCR cannot grade its own correctness - that needs a human-verified ground truth. Fill in "
         "the 'Session ID Correct? (Y/N)' and 'Date/Time Correct? (Y/N)' columns by eye against the "
         "zoomed images (Y = the extracted value is fully correct, N = it has any error), save the "
         "file, then re-run the script with --evaluate-only (see the top of the .py file). It will "
         "print Accuracy, a Confusion Matrix, and Type I / Type II error rates for each field, "
         "treating 'Manual Review? = YES' as the model's predicted-error flag and your Y/N as the "
         "actual outcome, and it will add an 'Accuracy Report' sheet to this workbook.", notes_font),
    ]
    for ridx, (text, font) in enumerate(lines, start=1):
        cell = notes.cell(row=ridx, column=1, value=text)
        cell.font = font
        cell.alignment = Alignment(wrap_text=True, vertical="top")
    notes.column_dimensions["A"].width = 130
    for ridx in range(1, len(lines) + 1):
        notes.row_dimensions[ridx].height = 30 if len(lines[ridx - 1][0]) > 80 else 18

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    wb.save(out_path)
    print("Saved Excel file:", out_path)


# =====================================================================
# ACCURACY / CONFUSION MATRIX / TYPE I & II ERRORS
# =====================================================================
# "Predicted error" = the row's Manual Review flag was YES for that field
# (session_needs_review / datetime_needs_review, recomputed here from the
# 'Manual Review?' + flags/date/time cells already in the sheet).
# "Actual error"    = you marked the Correct? column N (i.e. OCR was wrong).
#
# Type I error  (false positive) = flagged for review, but was actually correct
#                                   -> wastes reviewer time, not a safety issue.
# Type II error (false negative) = NOT flagged, but was actually wrong
#                                   -> the dangerous case: a bad value slipped
#                                   through unnoticed.

def _confusion_and_rates(pairs):
    """pairs: list of (predicted_flagged: bool, actual_wrong: bool)"""
    tp = sum(1 for p, a in pairs if p and a)        # flagged & actually wrong
    fp = sum(1 for p, a in pairs if p and not a)     # flagged & actually correct  -> Type I
    fn = sum(1 for p, a in pairs if not p and a)     # not flagged & actually wrong -> Type II
    tn = sum(1 for p, a in pairs if not p and not a)  # not flagged & actually correct
    total = len(pairs)
    accuracy = (tp + tn) / total if total else 0.0
    type1_rate = fp / (fp + tn) if (fp + tn) else 0.0   # false positive rate
    type2_rate = fn / (fn + tp) if (fn + tp) else 0.0   # false negative rate
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn, "total": total,
            "accuracy": accuracy, "type1_rate": type1_rate, "type2_rate": type2_rate}


def evaluate_accuracy(excel_path=None):
    excel_path = excel_path or OUTPUT_EXCEL
    wb = load_workbook(excel_path)
    ws = wb["CDS Audit Results"]
    headers = [c.value for c in ws[1]]
    col = {h: i + 1 for i, h in enumerate(headers)}

    session_pairs, datetime_pairs = [], []
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        flags = (row[col["Char Flags / Notes"] - 1].value or "")
        date_v = row[col["Date"] - 1].value
        time_v = row[col["Time"] - 1].value
        session_flagged = "LOW CONFIDENCE" in flags
        datetime_flagged = not datetime_plausible(date_v, time_v)

        s_correct = row[col["Session ID Correct? (Y/N)"] - 1].value
        d_correct = row[col["Date/Time Correct? (Y/N)"] - 1].value

        if s_correct and str(s_correct).strip().upper() in ("Y", "N"):
            actual_wrong = str(s_correct).strip().upper() == "N"
            session_pairs.append((session_flagged, actual_wrong))
        if d_correct and str(d_correct).strip().upper() in ("Y", "N"):
            actual_wrong = str(d_correct).strip().upper() == "N"
            datetime_pairs.append((datetime_flagged, actual_wrong))

    report_lines = []
    report_lines.append("CDS Audit - Accuracy Report")
    report_lines.append("=" * 40)

    for label, pairs in [("Session ID", session_pairs), ("Date/Time", datetime_pairs)]:
        report_lines.append("")
        report_lines.append(f"--- {label} ---")
        if not pairs:
            report_lines.append(f"No rows have '{label} Correct? (Y/N)' filled in yet - skipped.")
            continue
        m = _confusion_and_rates(pairs)
        report_lines.append(f"Rows evaluated: {m['total']}")
        report_lines.append(f"Accuracy: {m['accuracy']*100:.1f}%")
        report_lines.append("Confusion Matrix (rows = predicted, cols = actual):")
        report_lines.append(f"                    Actual Wrong   Actual Correct")
        report_lines.append(f"  Flagged (Review)   TP={m['tp']:<10} FP={m['fp']:<10}  <- Type I errors")
        report_lines.append(f"  Not Flagged        FN={m['fn']:<10} TN={m['tn']:<10}  <- Type II errors")
        report_lines.append(f"Type I error rate  (correct rows wrongly flagged for review): {m['type1_rate']*100:.1f}%")
        report_lines.append(f"Type II error rate (wrong rows that slipped through unflagged): {m['type2_rate']*100:.1f}%")

    report_text = "\n".join(report_lines)
    print(report_text)

    if "Accuracy Report" in wb.sheetnames:
        del wb["Accuracy Report"]
    rs = wb.create_sheet("Accuracy Report")
    mono = Font(name="Consolas", size=10)
    for i, line in enumerate(report_text.splitlines(), start=1):
        rs.cell(row=i, column=1, value=line).font = mono
    rs.column_dimensions["A"].width = 90
    wb.save(excel_path)
    print("\nAccuracy Report sheet added to:", excel_path)


# =====================================================================
if __name__ == "__main__":
    if "--evaluate-only" in sys.argv:
        evaluate_accuracy()
    else:
        print("Tesseract version:", pytesseract.get_tesseract_version())
        all_records = run_batch()
        build_excel(all_records, OUTPUT_EXCEL)
        print("\nALL DONE. Open:", OUTPUT_EXCEL)
        print("\nNext: fill in the 'Session ID Correct? (Y/N)' and 'Date/Time Correct? (Y/N)' "
              "columns by eye, save the file, then run this script again with --evaluate-only "
              "to get Accuracy / Confusion Matrix / Type I & II errors.")

Tesseract version: 4.1.1
Found 40 images. Starting...
[1/40] day 1 batches / 20260612_152601.jpg
[2/40] day 1 batches / 20260612_152609.jpg
[3/40] day 1 batches / 20260612_152614.jpg
[4/40] day 1 batches / 20260612_152619.jpg
[5/40] day 1 batches / 20260612_152624.jpg
[6/40] day 1 batches / 20260612_152629.jpg
[7/40] day 1 batches / 20260612_152633.jpg
[8/40] day 1 batches / 20260612_152644.jpg
[9/40] day 1 batches / 20260612_152649.jpg
[10/40] day 1 batches / 20260612_152657.jpg
[11/40] day 1 batches / 20260612_152711.jpg
[12/40] day 1 batches / 20260612_152723.jpg
[13/40] day 2 batches / 20260612_151444.jpg
[14/40] day 2 batches / 20260612_151451.jpg
[15/40] day 2 batches / 20260612_151457.jpg
[16/40] day 2 batches / 20260612_151507.jpg
[17/40] day 2 batches / 20260612_151513.jpg
[18/40] day 2 batches / 20260612_151518.jpg
[19/40] day 2 batches / 20260612_151525.jpg
[20/40] day 2 batches / 20260612_151531.jpg
[21/40] day 2 batches / 20260612_151538.jpg
[22/40] day 2 batches / 2026061